In [ ]:
# Google Colab only — uncomment and run if your dataset is in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

# Change this to the folder containing your train/ and valid/ (or val/) folders.
# Example for Google Drive:
# DATASET_DIR = "/content/drive/MyDrive/TomatoDataset"
DATASET_DIR = "/content/TomatoDataset"

TRAIN_DIR = os.path.join(DATASET_DIR, "train")
VAL_DIR = os.path.join(DATASET_DIR, "valid")
if not os.path.isdir(VAL_DIR):
    VAL_DIR = os.path.join(DATASET_DIR, "val")

print("Train directory:", TRAIN_DIR, "| exists:", os.path.isdir(TRAIN_DIR))
print("Validation directory:", VAL_DIR, "| exists:", os.path.isdir(VAL_DIR))

if not os.path.isdir(TRAIN_DIR) or not os.path.isdir(VAL_DIR):
    print("\nUpdate DATASET_DIR so it points to a folder containing train/ and valid/ (or val/).")

In [ ]:
# If a package is missing in your environment, uncomment:
# %pip install torch torchvision matplotlib scikit-learn

import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
IMAGE_SIZE = 128
BATCH_SIZE = 32
NUM_WORKERS = 2 if os.name != "nt" else 0

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

if os.path.isdir(TRAIN_DIR) and os.path.isdir(VAL_DIR):
    train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
    val_dataset = datasets.ImageFolder(VAL_DIR, transform=eval_transform)

    class_names = train_dataset.classes
    if class_names != val_dataset.classes:
        raise ValueError(
            "Train and validation class folders do not match. "
            f"Train: {class_names}; validation: {val_dataset.classes}"
        )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"))

    print("Classes:", class_names)
    print("Training images:", len(train_dataset))
    print("Validation images:", len(val_dataset))
else:
    print("Dataset folders not found yet. Set DATASET_DIR correctly, then rerun this cell.")

## 6. Preview sample images

In [ ]:
if 'train_dataset' in globals():
    images, labels = next(iter(train_loader))
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
        image = (image * std + mean).clamp(0, 1)
        ax.imshow(image.permute(1, 2, 0))
        ax.set_title(class_names[label])
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Load the dataset first.")

In [ ]:
class TomatoCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.20),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

if 'class_names' in globals():
    model = TomatoCNN(num_classes=len(class_names)).to(device)
    print(model)

In [ ]:
EPOCHS = 10  # Increase after confirming the notebook runs correctly
LEARNING_RATE = 1e-3

if 'train_loader' in globals():
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_loss = float("inf")
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(EPOCHS):
        start_time = time.time()

        model.train()
        train_loss_sum, train_correct, train_total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * inputs.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += labels.size(0)

        train_loss = train_loss_sum / max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)

        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss_sum += loss.item() * inputs.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_sum / max(val_total, 1)
        val_acc = val_correct / max(val_total, 1)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())

        print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
              f"train loss {train_loss:.4f}, acc {train_acc:.3%} | "
              f"val loss {val_loss:.4f}, acc {val_acc:.3%} | "
              f"{time.time()-start_time:.1f}s")

    model.load_state_dict(best_weights)
else:
    print("Load the dataset and create the model before training.")

In [ ]:
if 'history' in globals() and history["train_loss"]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Validation")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history["train_acc"], label="Train")
    axes[1].plot(history["val_acc"], label="Validation")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("Train the model first.")

In [ ]:
if 'val_loader' in globals() and 'model' in globals():
    y_true, y_pred = [], []
    model.eval()
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs.to(device))
            y_pred.extend(outputs.argmax(1).cpu().numpy())
            y_true.extend(labels.numpy())

    print(classification_report(y_true, y_pred, target_names=class_names, digits=4,
                                zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(max(8, len(class_names)), max(6, len(class_names)*0.7)))
    display.plot(ax=ax, xticks_rotation=90, colorbar=False)
    plt.tight_layout()
    plt.show()
else:
    print("Load the dataset and train the model first.")

In [ ]:
if 'model' in globals() and 'class_names' in globals():
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "class_names": class_names,
        "image_size": IMAGE_SIZE,
        "architecture": "TomatoCNN"
    }
    torch.save(checkpoint, "tomato_leaf_disease_cnn.pth")
    print("Saved: tomato_leaf_disease_cnn.pth")
else:
    print("Train the model first.")

In [ ]:
from PIL import Image

def predict_leaf(image_path, model, class_names):
    image = Image.open(image_path).convert("RGB")
    tensor = eval_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        probabilities = torch.softmax(model(tensor), dim=1)[0]
        confidence, predicted_idx = probabilities.max(dim=0)

    predicted_class = class_names[predicted_idx.item()]
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Prediction: {predicted_class}\nModel confidence: {confidence.item():.2%}")
    plt.show()

    return predicted_class, confidence.item()

# Example:
# prediction, confidence = predict_leaf("/path/to/tomato_leaf.jpg", model, class_names)
# print("Predicted class:", prediction, "| Confidence:", f"{confidence:.2%}")